# Violent Crime Capstone



**Main question:** How well can socioeconomic factors predict violent crime rates?

**Subquestion:** Which categories are the strongest predictors of violent crime?



## 1. Imports and notebook setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

#import sys
#print(sys.executable)


/opt/miniconda3/bin/python


In [12]:
from pathlib import Path

BASE_DIR = Path.cwd()
CLEAN_DATA = BASE_DIR   / 'cleaned_data'
OUTPUTS = BASE_DIR  / 'outputs'

CLEAN_DATA.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

crime_file = BASE_DIR   / 'CIUS_Table_10_Offenses_Known_to_Law_Enforcement_by_State_by_Metropolitan_and_Nonmetropolitan_Counties_2024.xlsx'

crime = pd.read_excel(crime_file, header=4)


crime.columns = (
    crime.columns.astype(str)
    .str.strip()
    .str.replace('\n', '_', regex=False)
    .str.replace(' ', '_', regex=False)
    .str.replace('/', '_', regex=False)
    .str.replace('-', '_', regex=False)
    .str.replace(r'[^A-Za-z0-9_]', '', regex=True)
    .str.lower()
)
crime.columns

Index(['state', 'metropolitan_nonmetropolitan', 'county', 'violent_crime',
       'murder_and_nonnegligent_manslaughter', 'rape', 'robbery',
       'aggravated_assault', 'property_crime', 'burglary', 'larceny__theft',
       'motor_vehicle_theft', 'arson1'],
      dtype='object')

In [13]:
crime = crime.rename(columns={
    'larceny__theft': 'larceny_theft',
    'arson1': 'arson'
})

crime.columns


Index(['state', 'metropolitan_nonmetropolitan', 'county', 'violent_crime',
       'murder_and_nonnegligent_manslaughter', 'rape', 'robbery',
       'aggravated_assault', 'property_crime', 'burglary', 'larceny_theft',
       'motor_vehicle_theft', 'arson'],
      dtype='object')

In [16]:
crime.isna().sum()


state                                   0
metropolitan_nonmetropolitan            2
county                                  2
violent_crime                           2
murder_and_nonnegligent_manslaughter    2
rape                                    2
robbery                                 2
aggravated_assault                      2
property_crime                          2
burglary                                2
larceny_theft                           2
motor_vehicle_theft                     2
arson                                   4
dtype: int64

In [17]:
crime[['state', 'county']].head(20)

,state,county
0,ALABAMA,Autauga
1,ALABAMA,Baldwin
2,ALABAMA,Bibb
3,ALABAMA,Blount
4,ALABAMA,Calhoun
5,ALABAMA,Chilton
6,ALABAMA,Colbert
7,ALABAMA,Elmore
8,ALABAMA,Etowah
9,ALABAMA,Geneva


In [18]:
crime[crime.isna().any(axis=1)]

,state,metropolitan_nonmetropolitan,county,violent_crime,murder_and_nonnegligent_manslaughter,rape,robbery,aggravated_assault,property_crime,burglary,larceny_theft,motor_vehicle_theft,arson
1377,NEW YORK,Metropolitan County,Nassau,"1,744.000",10.000,140.000,396.000,"1,198.000","11,887.000",832.000,"10,767.000",288.000,NaN
1400,NEW YORK,Nonmetropolitan County,Delaware,12.000,0.000,6.000,0.000,6.000,52.000,6.000,41.000,5.000,NaN
2444,1 The FBI does not publish arson data unless i...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2445,2 Limited data for 2024 were available for Flo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
crime = crime.dropna(subset=['county', 'violent_crime']).copy()
crime['arson'] = crime['arson'].fillna(0)
crime_cols = [
    'violent_crime',
    'murder_and_nonnegligent_manslaughter',
    'rape',
    'robbery',
    'aggravated_assault',
    'property_crime',
    'burglary',
    'larceny_theft',
    'motor_vehicle_theft',
    'arson'
]

crime[crime_cols] = crime[crime_cols].apply(pd.to_numeric, errors='coerce')
crime.isna().sum()
crime.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2444 entries, 0 to 2443
Data columns (total 13 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   state                                 2444 non-null   object 
 1   metropolitan_nonmetropolitan          2444 non-null   object 
 2   county                                2444 non-null   object 
 3   violent_crime                         2444 non-null   float64
 4   murder_and_nonnegligent_manslaughter  2444 non-null   float64
 5   rape                                  2444 non-null   float64
 6   robbery                               2444 non-null   float64
 7   aggravated_assault                    2444 non-null   float64
 8   property_crime                        2444 non-null   float64
 9   burglary                              2444 non-null   float64
 10  larceny_theft                         2444 non-null   float64
 11  motor_vehicle_theft   

In [21]:
crime[crime_cols] = crime[crime_cols].astype(int)
crime.info()


<class 'pandas.core.frame.DataFrame'>
Index: 2444 entries, 0 to 2443
Data columns (total 13 columns):
 #   Column                                Non-Null Count  Dtype 
---  ------                                --------------  ----- 
 0   state                                 2444 non-null   object
 1   metropolitan_nonmetropolitan          2444 non-null   object
 2   county                                2444 non-null   object
 3   violent_crime                         2444 non-null   int64 
 4   murder_and_nonnegligent_manslaughter  2444 non-null   int64 
 5   rape                                  2444 non-null   int64 
 6   robbery                               2444 non-null   int64 
 7   aggravated_assault                    2444 non-null   int64 
 8   property_crime                        2444 non-null   int64 
 9   burglary                              2444 non-null   int64 
 10  larceny_theft                         2444 non-null   int64 
 11  motor_vehicle_theft                

In [23]:
crime['year'] = 2024

In [24]:
crime.to_csv(CLEAN_DATA / 'crime_cleaned.csv', index=False)
crime.head

<bound method NDFrame.head of          state metropolitan_nonmetropolitan      county  violent_crime  \
0      ALABAMA          Metropolitan County     Autauga             54   
1      ALABAMA          Metropolitan County     Baldwin            137   
2      ALABAMA          Metropolitan County        Bibb             34   
3      ALABAMA          Metropolitan County      Blount             82   
4      ALABAMA          Metropolitan County     Calhoun            238   
...        ...                          ...         ...            ...   
2439  WYOMING        Nonmetropolitan County  Sweetwater             11   
2440  WYOMING        Nonmetropolitan County       Teton              5   
2441  WYOMING        Nonmetropolitan County       Uinta              4   
2442  WYOMING        Nonmetropolitan County    Washakie              2   
2443  WYOMING        Nonmetropolitan County      Weston              2   

      murder_and_nonnegligent_manslaughter  rape  robbery  aggravated_assault  \
